In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 5.11 A Taste of Non-Equilibrium: Irreversibility, the Arrow of Time, and the Approach to Equilibrium

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume V — Classical Statistical Mechanics",
    number="5.11",
    title="A Taste of Non-Equilibrium: Irreversibility, the Arrow of Time, and the Approach to Equilibrium",
    blurb="The question every ensemble quietly assumed. Microscopic dynamics "
    "runs equally well backwards and returns to its start, yet a gas spreads, "
    "heat flows, and entropy rises — once. We watch irreversibility emerge from "
    "reversible rules in a ring of flipping balls, see a gas find the "
    "Maxwell–Boltzmann law, and meet Brownian motion and the Einstein relation. "
    "This is a glimpse through a door into a subject of its own; we open it just "
    "wide enough to see in.",
    difficulty="advanced",
    estimate="220–260 min",
)

## Notebook overview

A frank word at the outset, because this notebook is unlike the others. It is the longest and most
demanding of the volume, and deliberately so — and it is, we think, among the most rewarding of the
whole series. The reason is honest scope: non-equilibrium statistical mechanics is genuinely a
subject of its own, several courses' worth, and we do not attempt a survey. We attempt a single
coherent **taste**. The title word is load-bearing. We will follow *one* thread — irreversibility,
the arrow of time — as far as a notebook can honestly take it, and we will name the rest of the
field's great landmarks as **horizons**, pointing through the door without pretending to walk the
whole room. Modelling that honesty, about what one sitting can and cannot do, is part of the lesson.

The thread answers a question the entire volume quietly assumed and never asked. Every ensemble we
built — microcanonical, canonical, grand canonical — *described* equilibrium: where a system ends
up. None asked *why* it goes there. And the puzzle, once you see it, is sharp. The microscopic
dynamics is **time-reversible** (Hamilton's equations run backwards just as well; Liouville's
theorem of [§5.5](ergodicity.ipynb)) and **recurrent** (Poincaré: a bounded system returns arbitrarily close to any past
state). Yet a gas released in a corner spreads to fill its box and never spontaneously gathers back;
heat flows from hot to cold and not the reverse; entropy rises in *one* direction. How can a
temporal asymmetry — an arrow of time — emerge from laws that have none?

The resolution, which we will *show* rather than assert, is that irreversibility is **emergent and
probabilistic**. It is not in the microscopic laws; it appears when we coarse-grain and count. The
**Kac ring** makes this exactly computable: a deterministic, reversible, periodic toy in which a
coarse-grained quantity nonetheless relaxes monotonically — Loschmidt's reversal and Zermelo's
recurrence objections to Boltzmann, made concrete and answered in a single model. A gas of colliding
particles then shows **Boltzmann's H-theorem** in action, relaxing irreversibly to the very
Maxwell–Boltzmann law of [§5.6](ideal-gas-fundamental-relation.ipynb). And the deep reason is the one that has run through the whole volume:
the equilibrium macrostate is *overwhelmingly* the most probable (the $1/\sqrt N$ concentration of
[§5.3](large-n-limit.ipynb)), so a system started elsewhere almost certainly moves toward it, while the reversed,
entropy-decreasing trajectories — which genuinely exist — are vanishingly rare, and the recurrence
times, though real, are longer than the age of the universe by unimaginable factors.

We end with a first taste of **transport** — Brownian motion, the Langevin equation, and the
Green–Kubo relation — which is the *dynamical* face of the fluctuation–response idea of [§5.9](grand-canonical-ensemble-equivalence.ipynb). There,
[§5.9](grand-canonical-ensemble-equivalence.ipynb) told us a response function is a static fluctuation; here a transport coefficient turns out to be
an equilibrium *time* correlation, and the Einstein relation $D=kT/\gamma$ ties diffusion to friction
as two faces of the same molecular collisions. The loop closes: the non-equilibrium dynamics relaxes
to exactly the Boltzmann equilibrium the rest of the volume described. We began the volume by
counting microstates; we end it by watching the arrow of time turn out to be a count.

> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: a reversible trajectory retracing itself; the Kac ring relaxing as $(1-2\mu)^t$ and then
> *recurring* exactly at $t=2N$; the Maxwell–Boltzmann distribution minimizing $H$; a gas relaxing
> to it with $H$ monotone; equilibrium's overwhelming probability and astronomical recurrence; free
> Brownian motion with $\langle x^2\rangle=2Dt$ and $D=kT/\gamma$; the velocity autocorrelation
> integrating (Green–Kubo) to that same $D$; and a trapped particle relaxing to $\langle x^2\rangle
> =kT/k$. A ✓ is strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope — a taste, and the horizons.** One thread (irreversibility) is developed in full; the rest
> of non-equilibrium statistical mechanics is *named, not derived* — the full Boltzmann transport
> equation, the Green–Kubo relations for viscosity and conductivity, Onsager reciprocity, linear
> response and the dynamical fluctuation–dissipation theorem, the Fokker–Planck and master equations,
> hydrodynamics, and the modern fluctuation theorems (Jarzynski, Crooks). The production craft of
> stochastic simulation belongs to molecular-simulation courses and MMM. See Kac (1959); Kardar,
> *Statistical Physics of Particles*; Chandler; Zwanzig, *Nonequilibrium Statistical Mechanics*; and
> Notebooks [§5.5](ergodicity.ipynb) (Liouville, reversibility), [§5.6](ideal-gas-fundamental-relation.ipynb) (Maxwell–Boltzmann), [§5.3](large-n-limit.ipynb) ($1/\sqrt N$), [§5.4](microstates-entropy-temperature.ipynb) (the
> Boltzmann equilibrium), [§5.9](grand-canonical-ensemble-equivalence.ipynb) (fluctuation–response).

## Theory in brief

### The puzzle: irreversibility from reversible laws

The microscopic dynamics is **time-reversible** and **recurrent**,

```{math}
:label: eq-puzzle
\text{reverse all velocities} \Rightarrow \text{the system retraces its path};\qquad
\text{Poincaré: it returns arbitrarily close to any past state,}
```

yet macroscopic systems approach equilibrium irreversibly and entropy rises in one direction. How a
temporal asymmetry emerges from time-symmetric laws is the founding question of non-equilibrium
statistical mechanics — the one the equilibrium theory of [§5.4](microstates-entropy-temperature.ipynb)–[§5.10](ising-emergence-universality.ipynb) never asked.

### The Kac ring: irreversibility made exactly computable

$N$ balls (black or white) sit on a ring; **stationary** markers occupy a fraction $\mu$ of the
edges. Each step every ball steps one site and flips colour iff it crosses a marker. The dynamics is
deterministic, **reversible**, and **periodic** (period $2N$: after two circuits every marker is
crossed twice, restoring every colour — Poincaré recurrence, exactly). Yet under the
**molecular-chaos** assumption (treat the markers a ball is about to meet as a random fraction
$\mu$), the order parameter $\Delta=\frac1N\sum_i\sigma_i$ relaxes monotonically,

```{math}
:label: eq-kac
\Delta(t)=(1-2\mu)^t\,\Delta(0)\ \to\ 0\ \text{(relaxation)},\qquad \Delta(2N)=\Delta(0)\ \text{(recurrence)} .
```

The same model shows *irreversible* relaxation (coarse-grained, probabilistic) and *exact*
reversibility/recurrence — Loschmidt and Zermelo, answered: irreversibility is emergent and
approximate, not fundamental.

### The Boltzmann equation and the H-theorem

Boltzmann's kinetic equation evolves a gas's velocity distribution $f(\mathbf v,t)$ under collisions.
Its decisive input is the **Stoßzahlansatz** (molecular chaos): colliding particles are *uncorrelated
before* they collide. That assumption — not the microscopic dynamics — breaks time-reversal symmetry,
and from it follows the **H-theorem**,

```{math}
:label: eq-htheorem
H=\int f\ln f\,d\mathbf v\ \text{ decreases monotonically until } f=f_{\rm MB},
```

the entropy $-kH$ rising to its maximum at the unique stationary state, Maxwell–Boltzmann. The full
collision integral and transport equation are *named, not derived* (a horizon).

### Relaxation to Maxwell–Boltzmann

We demonstrate the H-theorem with a caricature of the Boltzmann equation: repeated random binary
collisions that conserve momentum and energy (rotate each colliding pair's relative velocity by a
random angle — molecular chaos made literal). From a non-equilibrium start the speed distribution
relaxes to the two-dimensional **Maxwell–Boltzmann** law ([§5.6](ideal-gas-fundamental-relation.ipynb)),

```{math}
:label: eq-relaxation
f(v)=\frac{m}{kT}\,v\,e^{-mv^2/2kT},\qquad H\ \text{monotone decreasing,}
```

and the equilibrium it finds is exactly the Boltzmann distribution of [§5.4](microstates-entropy-temperature.ipynb)/[§5.6](ideal-gas-fundamental-relation.ipynb).

### The resolution of the paradoxes

Irreversibility does not contradict reversibility; it emerges from **coarse-graining + overwhelming
probability**,

```{math}
:label: eq-resolution
\frac{\#\,\text{equilibrium microstates}}{\#\,\text{all microstates}}\to1,\qquad t_{\rm recurrence}\sim e^{O(N)} .
```

The equilibrium macrostate occupies almost all of phase space (the $1/\sqrt N$ concentration of [§5.3](large-n-limit.ipynb)),
so a system almost certainly moves toward it; reversed trajectories exist but are vanishingly rare,
and recurrence times, though real, are astronomically long. The arrow of time is the statistics of
large numbers applied to dynamics.

### A taste of transport: Brownian motion and Green–Kubo

A heavy particle in a fluid obeys the **Langevin** equation $m\,\dot v=-\gamma v+\xi(t)$ with
$\langle\xi(t)\xi(t')\rangle=2\gamma kT\,\delta(t-t')$ — friction and noise are two faces of the same
collisions (Einstein 1905). Free Brownian motion diffuses, $\langle x^2\rangle=2Dt$, with the
**Einstein relation** $D=kT/\gamma$. The velocity **autocorrelation function**
$C(t)=\langle v(0)v(t)\rangle$ — how long a fluctuating quantity remembers itself — and the
**Green–Kubo** relation give the transport coefficient as its time integral,

```{math}
:label: eq-transport
C(t)=\frac{kT}{m}e^{-(\gamma/m)t},\qquad D=\int_0^\infty\!\langle v(0)v(t)\rangle\,dt=\frac{kT}{\gamma} .
```

Transport coefficients *are* equilibrium time-correlation functions — the **dynamical**
fluctuation–dissipation theorem, the time-resolved extension of the static response = fluctuation of [§5.9](grand-canonical-ensemble-equivalence.ipynb).

### The horizons

What this one notebook only peeked at, each a pointer and none developed: the full **Boltzmann
transport equation** and kinetic theory; the **Green–Kubo** relations for viscosity and thermal
conductivity; **Onsager**'s reciprocal relations; **linear response** and the dynamical
fluctuation–dissipation theorem in full; the **Fokker–Planck** and master equations; hydrodynamics;
and the modern **fluctuation theorems** (Jarzynski's $\langle e^{-\beta W}\rangle=e^{-\beta\Delta F}$,
Crooks) relating non-equilibrium work to equilibrium free energy. Non-equilibrium
statistical mechanics is a subject of its own; this was a taste.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Units throughout: k_B = 1. Masses, frictions, temperatures stated per exercise.


def kac_ring(N, mu, n_steps, rng):
    """Simulate the Kac ring and return its order-parameter trace Δ(t) = (1/N)·Σ_i σ_i.

    N balls σ_i = ±1 on a ring; **stationary** markers η_i ∈ {+1, −1} on the edges
    (a fraction μ are −1, "flip"). Each step the balls circulate one site (``numpy.roll``) and
    each flips colour across a marker: σ(t+1) = η·roll(σ, 1). The
    markers do **not** move — that is the subtle point of the model, and the easy bug. The dynamics is
    deterministic, reversible, and periodic with period 2N.

    Parameters
    ----------
    N : int
        Number of balls/sites.
    mu : float
        Fraction of edges carrying a (flipping) marker.
    n_steps : int
        Number of steps to evolve.
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``) — used only to *place* the markers.

    Returns
    -------
    numpy.ndarray
        The order-parameter trace Δ(t), length ``n_steps + 1``.
    """
    sigma = np.ones(N, dtype=int)  # all-white start, Δ(0) = 1
    eta = np.where(rng.random(N) < mu, -1, 1)  # stationary markers (−1 = flip)
    delta = [sigma.mean()]
    for _ in range(n_steps):
        sigma = np.roll(sigma, 1) * eta  # circulate one site and flip across markers
        delta.append(sigma.mean())
    return np.array(delta)


def H_functional(speeds, bins=30, v_max=3.5):
    """Boltzmann's H = ∫ f ln f dv, estimated from a speed sample by histogram (`numpy.histogram`).

    Forms the normalized speed density f on a fixed grid and returns Σ_i f_i ln f_i Δv
    over the occupied bins — the discretized ∫ f ln f. The H-theorem says this decreases as a
    gas relaxes, reaching its minimum at the Maxwell–Boltzmann distribution.

    Parameters
    ----------
    speeds : numpy.ndarray
        Particle speeds.
    bins : int, optional
        Number of histogram bins (default ``30``).
    v_max : float, optional
        Upper edge of the speed grid (default ``3.5``).

    Returns
    -------
    float
        The estimated H.
    """
    f, edges = np.histogram(speeds, bins=bins, range=(0.0, v_max), density=True)
    dv = edges[1] - edges[0]
    nz = f > 0
    return float(np.sum(f[nz] * np.log(f[nz])) * dv)


def random_collisions(v, n_collisions, rng):
    """Apply energy- and momentum-conserving random binary collisions to a 2-D velocity array.

    Each collision picks two particles and rotates their *relative* velocity by a random angle, leaving
    the centre-of-mass velocity (momentum) and the relative speed (energy) unchanged — the
    molecular-chaos assumption made literal. Modifies ``v`` in place and returns it.

    Parameters
    ----------
    v : numpy.ndarray, shape (n, 2)
        Particle velocities (modified in place).
    n_collisions : int
        Number of binary collisions to apply.
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``).

    Returns
    -------
    numpy.ndarray
        The updated velocity array.
    """
    n = len(v)
    pairs = rng.integers(0, n, (n_collisions, 2))
    angles = rng.uniform(0, 2 * np.pi, n_collisions)
    for c in range(n_collisions):
        i, j = pairs[c]
        v_cm = 0.5 * (v[i] + v[j])
        v_rel = v[i] - v[j]
        cs, sn = np.cos(angles[c]), np.sin(angles[c])
        v_rel = np.array(
            [cs * v_rel[0] - sn * v_rel[1], sn * v_rel[0] + cs * v_rel[1]]
        )  # rotate
        v[i] = v_cm + 0.5 * v_rel
        v[j] = v_cm - 0.5 * v_rel
    return v


def langevin_baoab(
    n_part, n_steps, dt, m, gamma, kT, rng, k_spring=0.0, x0=0.0, v_init=None
):
    """Integrate N Langevin particles with the **BAOAB** splitting and return their x, v histories.

    The Langevin equation m·dv/dt = −k_spring·x − γ·v + ξ is integrated by the symmetric
    BAOAB scheme — half a force kick (B), half a drift (A), the exact Ornstein–Uhlenbeck thermostat
    step (O, v ← c·v + √((1 − c^2)·kT/m)·N(0,1) with c = e^(−γ·dt/m)), half a
    drift, half a kick. BAOAB is used deliberately: a naive Euler–Maruyama integrator biases
    ⟨v^2⟩ away from kT/m and inflates the Green–Kubo integral, so one must verify
    ⟨v^2⟩ = kT/m before trusting the velocity-autocorrelation route.

    Parameters
    ----------
    n_part : int
        Number of independent particles.
    n_steps : int
        Number of timesteps.
    dt : float
        Timestep.
    m, gamma, kT : float
        Mass, friction coefficient, and temperature (k_B = 1).
    rng : numpy.random.Generator
        Random generator (``numpy.random.default_rng``).
    k_spring : float, optional
        Harmonic spring constant (default ``0.0`` for free Brownian motion).
    x0 : float, optional
        Initial position (default ``0.0``).
    v_init : float or None, optional
        Initial velocity; ``None`` draws from the Maxwell–Boltzmann √(kT/m)·N(0,1).

    Returns
    -------
    tuple of numpy.ndarray
        ``(X, V)`` each of shape ``(n_steps + 1, n_part)``.
    """
    x = np.full(n_part, x0, dtype=float)
    v = (
        np.sqrt(kT / m) * rng.standard_normal(n_part)
        if v_init is None
        else np.full(n_part, v_init, dtype=float)
    )
    c = np.exp(-gamma * dt / m)
    sigma = np.sqrt((1.0 - c * c) * kT / m)
    X = np.empty((n_steps + 1, n_part))
    V = np.empty((n_steps + 1, n_part))
    X[0], V[0] = x, v
    for t in range(n_steps):
        v += 0.5 * dt * (-k_spring * x) / m  # B: half kick
        x += 0.5 * dt * v  # A: half drift
        v = c * v + sigma * rng.standard_normal(n_part)  # O: exact OU thermostat
        x += 0.5 * dt * v  # A: half drift
        v += 0.5 * dt * (-k_spring * x) / m  # B: half kick
        X[t + 1], V[t + 1] = x, v
    return X, V


def vacf(V, n_lag):
    """The velocity autocorrelation function C(t) = ⟨v(0)v(t)⟩ from a velocity history.

    Averages over particles (the columns of ``V``) and over a block of time origins for variance
    reduction. The Green–Kubo integral of C is the diffusion coefficient.

    Parameters
    ----------
    V : numpy.ndarray, shape (n_steps + 1, n_part)
        Velocity history.
    n_lag : int
        Number of lag times to compute.

    Returns
    -------
    numpy.ndarray
        C(t) for lags 0, 1, …, n_lag − 1 (in timestep units).
    """
    n_origins = V.shape[0] - n_lag
    return np.array(
        [np.mean(V[:n_origins] * V[t : t + n_origins]) for t in range(n_lag)]
    )

## Exercise 1 — The puzzle of the arrow of time

We begin by sharpening the paradox until it is undeniable. The microscopic laws of motion have no
preferred direction of time: run Hamilton's equations backwards and they are still Hamilton's
equations, and Liouville's theorem of [§5.5](ergodicity.ipynb) says the flow conserves phase-space volume in either
direction. We can see the reversibility directly. Take a particle in some potential, integrate its
motion with a time-symmetric (velocity-Verlet) scheme, then *reverse its velocity* and integrate
again — it retraces its path exactly, arriving back where it started {eq}`eq-puzzle`
({numref}`fig-ne-puzzle`). And the dynamics is recurrent: Poincaré proved that a bounded system
returns arbitrarily close to any earlier state, given enough time. So at the level of the molecules,
nothing distinguishes forward from backward, and nothing is lost forever. Yet the world we see is
emphatically one-directional — gases spread, cream mixes into coffee, things cool, and never the
reverse. The whole equilibrium theory of this volume described the destination without ever asking
why the journey runs only one way. That question is the subject of this notebook.

**This exercise (worked).** Integrate a particle in an anharmonic potential with a reversible
velocity-Verlet scheme; reverse the velocity at the end and integrate the same number of steps;
confirm it returns to its exact starting point — the microscopic dynamics is time-reversible.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    [x_back, -v_back],
    [x_start, v_start],
    "reversing the velocities retraces the trajectory exactly — the microscopic dynamics is time-reversible (yet macroscopic behavior is not)",
    atol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The Kac ring I: irreversible relaxation under molecular chaos

To resolve the puzzle we need a model simple enough to understand completely and rich enough to show
the paradox, and Mark Kac's ring (1959) is the perfect one. Picture $N$ balls on a ring of $N$ sites,
each ball black or white, and fix **markers** on some fraction $\mu$ of the edges between sites
({numref}`fig-ne-kac-ring`). The rule: at every step each ball advances one site, and it flips
colour if and only if it crosses a marker. The markers never move; only the balls circulate. This
dynamics is utterly deterministic and, as we will see in Exercise 3, exactly reversible and periodic
— there is no randomness and no friction anywhere in it. And yet watch the order parameter
$\Delta=\frac1N\sum_i\sigma_i$ (the excess of one colour). If we make the **molecular-chaos
assumption** — that the markers a ball is about to encounter are a *random* fraction $\mu$ of the
edges ahead, uncorrelated with its history — then each step a fraction $\mu$ of balls flip, and a
short calculation gives monotonic, irreversible relaxation, $\Delta(t)=(1-2\mu)^t\Delta(0)\to0$
{eq}`eq-kac`. The coarse-grained quantity decays to equilibrium exactly as the H-theorem will have a
gas do — and from a perfectly reversible rule.

**This exercise (worked).** Implement the Kac ring (`kac_ring`, markers stationary, balls circulating
via `numpy.roll`); from an all-white start, track $\Delta(t)$, and confirm the early relaxation
matches the molecular-chaos prediction $(1-2\mu)^t$ and that $\Delta$ falls toward zero.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    delta[:8],
    mc_prediction,
    "under molecular chaos the Kac-ring order parameter relaxes as (1−2μ)^t — irreversible decay from a reversible rule",
    atol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — The Kac ring II: reversibility, recurrence, and the paradoxes resolved

Now we spring the trap that caught Boltzmann's critics, and disarm it. Boltzmann claimed his gas
relaxes irreversibly to equilibrium; two objections followed. **Loschmidt** (the reversal paradox):
the laws are time-symmetric, so for every relaxing trajectory there is a reversed one in which
entropy *decreases* — how can relaxation be universal? **Zermelo** (the recurrence paradox):
Poincaré says a bounded system returns to its start, so entropy cannot rise forever. The Kac ring
settles both, because we can compute its *exact* dynamics, not just the molecular-chaos
approximation. Run the ring out to $t=2N$ and the order parameter $\Delta$ snaps **exactly** back to
$\Delta(0)$ ({numref}`fig-ne-kac-relax`): after two full circuits every ball has crossed every marker
exactly twice, two flips cancel, and the original configuration is restored, period $2N$
{eq}`eq-kac`. And the dynamics is reversible — reverse the stepping and it retraces. So both
objections are *correct*: the exact dynamics is reversible and recurrent. They simply do not
contradict the relaxation, because the smooth $(1-2\mu)^t$ decay was never the exact dynamics — it was
the molecular-chaos *approximation*, which holds early (when the markers ahead really are
uncorrelated with a ball's history) and breaks down long before the recurrence. Irreversibility is
emergent and approximate, exact at neither the shortest nor the recurrence timescale. Loschmidt and
Zermelo, answered in one model.

**This exercise (worked).** Run the same ring to $t=2N$ and confirm $\Delta(2N)=\Delta(0)$ exactly
(Poincaré recurrence), with the molecular-chaos curve tracking only the early relaxation.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    recurrence_delta,
    delta[0],
    "the exact Kac-ring dynamics recurs at t=2N (Poincaré) — reversible and recurrent underneath the irreversible relaxation",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The H-theorem and the Stoßzahlansatz

With the Kac ring's lesson in hand, we name the real thing it models. Boltzmann's kinetic equation
describes how a gas's velocity distribution $f(\mathbf v,t)$ evolves under collisions, and its
crucial input is the **Stoßzahlansatz** — the molecular-chaos assumption that two particles about to
collide are *uncorrelated*, exactly the assumption that made the markers "random" in the Kac ring.
That assumption, and not the underlying reversible mechanics, is what breaks time-reversal symmetry,
and from it Boltzmann proved his **H-theorem**: the functional $H=\int f\ln f\,d\mathbf v$ decreases
monotonically (so the entropy $-kH$ increases) until $f$ reaches the unique stationary distribution
{eq}`eq-htheorem`. That stationary distribution is **Maxwell–Boltzmann** — and we can see *why* it is
special directly: among all velocity distributions with a fixed energy, Maxwell–Boltzmann is the one
that *minimizes* $H$ (equivalently, maximizes entropy), so any other same-energy distribution sits
higher and the H-theorem drives the gas downhill to it. The full collision integral and the Boltzmann
transport equation that surrounds it are a horizon we name but do not derive.

**This exercise (worked).** Confirm that the Maxwell–Boltzmann distribution minimizes $H$: sample it
and two other distributions at the *same* energy, compute $H$ for each (`H_functional`), and check
Maxwell–Boltzmann has the lowest.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    H_mb < H_equal and H_mb < H_bimodal,
    "the Maxwell–Boltzmann distribution has lower H than the tested non-equilibrium distributions — the equilibrium the H-theorem drives toward",
)

## Exercise 5 — A gas relaxes to Maxwell–Boltzmann

Now we watch the H-theorem happen. We model a gas by a caricature of the Boltzmann collision process:
repeated **random binary collisions** that conserve momentum and energy, each one rotating a
colliding pair's relative velocity by a random angle (`random_collisions`). This is molecular chaos
made literal — every collision scrambles the participants' correlations — and it is exactly the
ingredient that makes the dynamics irreversible. We start the gas in a sharply non-equilibrium state,
all particles moving at the *same* speed in random directions, and let the collisions run. The speed
distribution flows, irreversibly, to the two-dimensional **Maxwell–Boltzmann** law $f(v)\propto v\,
e^{-mv^2/2kT}$ of [§5.6](ideal-gas-fundamental-relation.ipynb) {eq}`eq-relaxation`, and Boltzmann's $H=\int f\ln f$ falls monotonically the
whole way ({numref}`fig-ne-relaxation`). Energy and momentum are conserved at every step, so nothing
is driving the gas but the statistics of collisions — and yet it has a clear, one-directional
destination. The equilibrium it finds is precisely the Boltzmann distribution the rest of the volume
computed from the ensembles; the dynamics and the ensembles agree.

**This exercise (worked).** Relax a gas from an all-equal-speed start by `random_collisions`, confirm
its speed distribution matches the 2-D Maxwell–Boltzmann law (the mean and rms speed), and confirm
$H$ decreased monotonically while the kinetic energy was conserved. The animation shows the
distribution morphing to Maxwell–Boltzmann.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    [speeds_final.mean(), np.sqrt((speeds_final**2).mean())],
    [np.sqrt(np.pi * kT_m / 2), np.sqrt(2 * kT_m)],
    "energy-conserving collisions relax the gas to the 2-D Maxwell–Boltzmann distribution (5.6)",
    rtol=2e-2,
)
validate.check(
    np.all(np.diff(H_history) < 0.01) and H_history[-1] < H_history[0] - 1,
    "H = ∫ f ln f decreases monotonically as the gas relaxes — the H-theorem in action",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The resolution: irreversibility as overwhelming probability

We can now state the resolution plainly, because every piece is in place. Irreversibility does not
contradict the reversibility of the microscopic laws; it emerges from **coarse-graining** plus
**overwhelming probability**, and the overwhelming probability is the same $1/\sqrt N$ concentration
that has run through the whole volume {eq}`eq-resolution`. The equilibrium macrostate corresponds to
so vastly many more microstates than any non-equilibrium one ([§5.3](large-n-limit.ipynb), [§5.4](microstates-entropy-temperature.ipynb)) that it occupies essentially
*all* of phase space; a system started in a rare, ordered corner therefore moves, with overwhelming
probability, toward equilibrium — not because it is forced to, but because almost every path leads
there. The reversed, entropy-*decreasing* trajectories that Loschmidt invoked genuinely exist, but
they are a vanishing fraction, and the Poincaré recurrences that Zermelo invoked are real but occur
on timescales of order $e^{N}$. Put a number on it: for a macroscopic gas of $N\sim10^{23}$
particles the recurrence time exceeds the age of the universe not by a large factor but by a factor
of order $10^{(10^{23})}$ — a number whose *exponent* dwarfs every quantity in cosmology. The arrow
of time is not a new law layered onto mechanics; it is the statistics of large numbers applied to
dynamics, the same sharpness that made macrostates definite in [§5.3](large-n-limit.ipynb) now making time's direction
effectively absolute.

**This exercise (worked).** Estimate the Poincaré recurrence time for a macroscopic system relative
to the age of the universe, and confirm it is astronomically large — recurrence and reversal are
real but utterly negligible.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    log10_recurrence_over_age > 1e20,
    "equilibrium is overwhelmingly probable and the recurrence time is astronomical (~e^N) — reversal and recurrence are real but negligible, so the arrow of time is effectively absolute",
)

## Exercise 7 — Brownian motion and the Einstein relation

Having understood *why* systems relax, we turn to *how fast* — transport — and take our first honest
taste of it through the phenomenon that founded the subject. A pollen grain in water jitters
ceaselessly, kicked by the molecules it cannot see; Einstein (1905) and Langevin (1908) modelled
this as a heavy particle obeying $m\,\dot v=-\gamma v+\xi(t)$, where the same molecular collisions
appear *twice* — as a systematic **friction** $-\gamma v$ and as a randomly fluctuating **force**
$\xi$ with $\langle\xi(t)\xi(t')\rangle=2\gamma kT\,\delta(t-t')$ {eq}`eq-transport`. That the
friction coefficient and the noise strength are locked together by the same $\gamma kT$ is the first
**fluctuation–dissipation relation**: dissipation (friction) and fluctuation (noise) are two faces of
one molecular reality. Its most famous consequence is that free Brownian motion **diffuses**,
$\langle x^2\rangle=2Dt$, with the **Einstein relation** $D=kT/\gamma$ ({numref}`fig-ne-brownian`) —
the diffusion constant (a fluctuation) fixed by the friction (a dissipation), the dynamical extension
of the static response = fluctuation of [§5.9](grand-canonical-ensemble-equivalence.ipynb). We integrate the Langevin equation with the BAOAB scheme,
whose accuracy we will need in the next exercise.

**This exercise (worked).** Simulate free Brownian particles with `langevin_baoab`, measure the
mean-square displacement $\langle x^2\rangle$, fit its diffusive slope $2D$, and confirm the Einstein
relation $D=kT/\gamma$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    D_msd,
    D_einstein,
    "free Brownian motion gives ⟨x²⟩ = 2Dt with the Einstein relation D = kT/γ (a fluctuation–dissipation relation)",
    rtol=8e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — Autocorrelation functions and Green–Kubo

We close the transport taste with the idea that is the modern language of the whole subject, and we
build the one new tool it needs from scratch, here where it is used. The tool is the
**autocorrelation function** $C(t)=\langle A(0)A(t)\rangle$: it measures how long a fluctuating
quantity *remembers itself*, starting at the variance $\langle A^2\rangle$ at $t=0$ and decaying over
a characteristic correlation time as the memory is lost. For the Brownian particle's velocity, the
**velocity autocorrelation function** is $C(t)=\langle v(0)v(t)\rangle=(kT/m)\,e^{-(\gamma/m)t}$, a
clean exponential decaying over the time $m/\gamma$ {eq}`eq-transport`. The remarkable **Green–Kubo**
relation then expresses the transport coefficient as the *time integral* of this equilibrium
correlation, $D=\int_0^\infty\langle v(0)v(t)\rangle\,dt=kT/\gamma$ ({numref}`fig-ne-vacf`) — the same
$D$ we got from the mean-square displacement and from Einstein, now as an integral over how the
velocity forgets itself. This is the deep, general statement: a transport coefficient *is* an
equilibrium time-correlation function, the **dynamical** fluctuation–dissipation theorem. One numerical
caution, which we honour: we first verify the integrator gives $\langle v^2\rangle=kT/m$ (an
inaccurate scheme biases this and inflates the integral), then integrate the autocorrelation with
`numpy.trapezoid`, truncating after a few correlation times.

**This exercise (worked).** Verify $\langle v^2\rangle=kT/m$ for the BAOAB trajectories; compute the
velocity autocorrelation `vacf`, confirm it decays as $(kT/m)e^{-(\gamma/m)t}$, and integrate it
(Green–Kubo, `numpy.trapezoid`) to recover $D=kT/\gamma$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    v2_mean,
    kT / m,
    "the BAOAB integrator gives ⟨v²⟩ = kT/m (verified before trusting the Green–Kubo integral)",
    rtol=2e-2,
)
validate.close(
    [C[0], D_greenkubo],
    [kT / m, kT / gamma],
    "the velocity autocorrelation integrates (Green–Kubo) to the diffusion coefficient D = kT/γ",
    rtol=5e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 9 — The Langevin dynamics relaxes to the Boltzmann equilibrium

Before the synthesis, we close the loop the whole volume has drawn — explicitly, with the dynamics
flowing into the ensembles. Put a Brownian particle not in free space but in a harmonic trap, $U(x)=
\tfrac12 k x^2$, and start it far from the bottom. The Langevin dynamics — the same friction and
noise — now does two things at once: the friction drains its excess energy and the noise keeps it
warm, and the competition settles into a stationary distribution. That distribution is exactly the
**Boltzmann** one, $P(x)\propto e^{-kx^2/2kT}$, with $\langle x^2\rangle=kT/k$ — equipartition, the
very result the canonical ensemble of [§5.8](partition-function.ipynb) gave for a harmonic degree of freedom. So the
non-equilibrium dynamics does not merely *resemble* the equilibrium theory; it *flows into* it. The
arrow of time we have been chasing points precisely at the equilibrium distributions the rest of the
volume built, and the two descriptions — the dynamics that takes a system there, and the ensemble
that describes it once arrived — are one consistent whole.

**This exercise (student).** Release a trapped Langevin particle from far off-centre, let it relax,
and confirm its equilibrium $\langle x^2\rangle=kT/k$ — the Boltzmann/equipartition result of the
ensembles, reached by the dynamics.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    x2_equilibrium,
    kT / k_trap,
    "the trapped Langevin particle relaxes to the Boltzmann equilibrium ⟨x²⟩ = kT/k (equipartition) — the dynamics flows into the ensembles",
    rtol=1e-1,
)

## Exercise 10 — A glimpse through the door, and the close of the volume

We have opened the door just wide enough to see that the room behind it is enormous. The equilibrium
theory of this volume described where systems rest; this notebook asked how they get there, and found
that irreversibility and the arrow of time are not extra laws but *emergent* facts — they appear,
probabilistically, the moment one coarse-grains and counts. The **Kac ring** made it exact: a
reversible, recurrent toy whose coarse-grained order parameter nonetheless relaxes, answering
Loschmidt and Zermelo in one model. A **gas finding Maxwell–Boltzmann** made it physical, the
H-theorem driving a non-equilibrium distribution downhill to the very equilibrium the ensembles
compute. The **resolution** named the mechanism: equilibrium is overwhelmingly probable (the
$1/\sqrt N$ of [§5.3](large-n-limit.ipynb) once more), recurrences and reversals real but astronomically rare. And **Brownian
motion, the Einstein relation, and Green–Kubo** gave a first taste of transport — diffusion and
friction as one molecular reality, a transport coefficient revealed as an equilibrium time-correlation,
the dynamical fluctuation–dissipation theorem extending the static one of [§5.9](grand-canonical-ensemble-equivalence.ipynb) — with the trapped
particle flowing back into the Boltzmann equilibrium and closing the loop.

And then the honest horizons, named not crossed: the full Boltzmann transport equation and kinetic
theory; the Green–Kubo relations for viscosity and conductivity; **Onsager** reciprocity; **linear
response** and the fluctuation–dissipation theorem in full; the **Fokker–Planck** and master
equations; hydrodynamics; and the modern **fluctuation theorems** of Jarzynski and Crooks, which tie
non-equilibrium work to equilibrium free energy. Each is a doorway of its own, and
non-equilibrium statistical mechanics is a subject of its own. This was a taste.

**This exercise (synthesis).** No new computation: the closing is the result. With it the **volume
closes**. We began at counting — how many microstates realize a macrostate ([§5.1](counting.ipynb)) — and built, from
probability alone, the entropy, the temperature, the ensembles, the structure of thermodynamics, the
emergence of order, and now the arrow of time. The arrow that felt like a fundamental law turned out
to be a *count*: the same statistics of large numbers that opened the volume, now pointing the
direction of time itself.

## Notebook summary

This final notebook of Volume V took a single honest taste of non-equilibrium statistical mechanics,
following irreversibility from reversible microscopic laws and closing the volume.

- **The puzzle** {eq}`eq-puzzle`: the microscopic dynamics is time-reversible (a Verlet trajectory
  retraces under velocity reversal) and recurrent, yet macroscopic systems relax irreversibly.
- **The Kac ring** {eq}`eq-kac`: a reversible, periodic toy whose order parameter relaxes as
  $(1-2\mu)^t$ under molecular chaos (verified) yet **recurs exactly** at $t=2N$ (verified) —
  Loschmidt's reversal and Zermelo's recurrence answered: irreversibility is emergent and approximate.
- **The H-theorem** {eq}`eq-htheorem`, {eq}`eq-relaxation`: the Stoßzahlansatz (molecular chaos)
  breaks time-reversal symmetry; Maxwell–Boltzmann minimizes $H=\int f\ln f$, and a gas of
  energy-conserving collisions relaxes to it (verified, $H$ monotone) — the Boltzmann equilibrium
  of [§5.6](ideal-gas-fundamental-relation.ipynb).
- **The resolution** {eq}`eq-resolution`: irreversibility is coarse-graining + overwhelming
  probability (the $1/\sqrt N$ of [§5.3](large-n-limit.ipynb)); recurrence times $\sim e^N$ dwarf the age of the universe by
  $\sim10^{(10^{23})}$.
- **A taste of transport** {eq}`eq-transport`: free Brownian motion gives $\langle x^2\rangle=2Dt$
  with the Einstein $D=kT/\gamma$; the velocity autocorrelation $(kT/m)e^{-(\gamma/m)t}$ integrates
  (Green–Kubo) to the same $D$ — transport coefficients are equilibrium time-correlations, the
  dynamical fluctuation–dissipation theorem ([§5.9](grand-canonical-ensemble-equivalence.ipynb)). A trapped particle relaxes to $\langle x^2\rangle
  =kT/k$, the ensembles' equilibrium.

Volume V built equilibrium from probability and then watched equilibrium assemble itself; the arrow
of time, which felt like a law, turned out to be the count that began the volume.

## Outlook

- **Non-equilibrium statistical mechanics, a subject of its own.** The Boltzmann transport equation
  and kinetic theory; the Green–Kubo relations for viscosity and thermal conductivity; Onsager
  reciprocity; linear response and the full fluctuation–dissipation theorem; the Fokker–Planck and
  master equations; hydrodynamics; and the modern fluctuation theorems (Jarzynski, Crooks) — each a
  horizon, a possible course of its own.
- **Stochastic and molecular simulation.** The Langevin integrator here is a first step; the
  production craft (thermostats, free-energy methods, rare events) belongs to molecular-simulation
  courses and MMM.
- **Volume VII: the quantum arrow of time.** Quantum dynamics, the quantum Boltzmann equation, and
  decoherence — how irreversibility appears in quantum systems — once Volume VI builds the mechanics.
- **Cross-reference** [§5.5](ergodicity.ipynb) (Liouville, reversibility, ergodicity), [§5.6](ideal-gas-fundamental-relation.ipynb) (Maxwell–Boltzmann), [§5.3](large-n-limit.ipynb)
  ($1/\sqrt N$), [§5.4](microstates-entropy-temperature.ipynb) (the Boltzmann equilibrium), [§5.9](grand-canonical-ensemble-equivalence.ipynb) (the static fluctuation–response this extends).

In [ ]:
from ecp.style import footer

footer()